# PitchCheck audio service on Kaggle

Runs `services/audio` (FastAPI + Gemma) on a Kaggle GPU and exposes it through a Cloudflare quick tunnel, so the web app can run on a laptop without a GPU.

Before running:

1. **Settings → Accelerator → GPU T4 x2**, and **Settings → Internet → On** (needs a phone-verified account).
2. **Add-ons → Secrets**: add `HF_TOKEN`, a Hugging Face token whose account has accepted the Gemma license.
3. Run all cells. The last one prints the tunnel URL. On the laptop:

```bash
AUDIO_SERVICE_URL=https://<printed-url> pnpm dev
```

The URL changes every time the tunnel starts, and the session stops when the notebook is idle or after about 12 hours.

In [ ]:
# audio input: google/gemma-4-E2B-it | google/gemma-4-E4B-it | google/gemma-4-12B-it
GEMMA_MODEL = "google/gemma-4-E2B-it"
BRANCH = "main"

In [ ]:
%cd /kaggle/working
!rm -rf pitchcheck && git clone -q --depth 1 -b {BRANCH} https://github.com/tomovicj/pitchcheck.git
%cd /kaggle/working/pitchcheck/services/audio

# Kaggle already has CUDA builds of torch and torchvision; requirements.txt would pull CPU wheels
!grep -vE '^(--extra-index-url|torch|torchvision)\b' requirements.txt > /tmp/requirements-kaggle.txt
!pip install -q -r /tmp/requirements-kaggle.txt
!which ffmpeg > /dev/null || (apt-get update -qq && apt-get install -y -qq ffmpeg > /dev/null)

In [ ]:
import os, subprocess, time, urllib.request
import torch
from kaggle_secrets import UserSecretsClient

assert torch.cuda.is_available(), "No GPU: set Settings -> Accelerator to a GPU"
env = {
    **os.environ,
    "HF_TOKEN": UserSecretsClient().get_secret("HF_TOKEN"),
    "GEMMA_MODEL": GEMMA_MODEL,
    # The T4 and P100 have no bf16 support
    "GEMMA_DTYPE": "auto" if torch.cuda.is_bf16_supported() else "float16",
    "HF_HOME": "/kaggle/working/hf",
}
print(torch.cuda.get_device_name(0), "->", env["GEMMA_DTYPE"])

server = subprocess.Popen(
    ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"],
    env=env, stdout=open("/kaggle/working/uvicorn.log", "w"), stderr=subprocess.STDOUT,
)

# The first start downloads the Gemma weights, which takes a few minutes
while True:
    if server.poll() is not None:
        print(open("/kaggle/working/uvicorn.log").read()[-3000:])
        raise RuntimeError("audio service exited")
    try:
        urllib.request.urlopen("http://localhost:8000/health")
        break
    except OSError:
        time.sleep(10)
print("audio service is up")

In [ ]:
import re

!test -x /kaggle/working/cloudflared || (curl -sL -o /kaggle/working/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /kaggle/working/cloudflared)
tunnel = subprocess.Popen(
    ["/kaggle/working/cloudflared", "tunnel", "--no-autoupdate", "--url", "http://localhost:8000"],
    stdout=open("/kaggle/working/tunnel.log", "w"), stderr=subprocess.STDOUT,
)

url = None
for _ in range(30):
    time.sleep(2)
    found = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/kaggle/working/tunnel.log").read())
    if found:
        url = found.group(0)
        break
print(f"AUDIO_SERVICE_URL={url}" if url else open("/kaggle/working/tunnel.log").read())

Service log, for debugging a run:

In [ ]:
!tail -n 50 /kaggle/working/uvicorn.log